# Basic Lanelet Abstraction Tutorial

For creating datasets of End2End driving with the existing Autoware-lanelet2 map, there are multiple functions we need to achieve

- lanelet2 loading with the correct projection
- extracting lanelets within a range of the search point (vehicle's current location).

In this tutorial, we will be loading the map from XX1 Odaiba map as an example

https://evaluation.tier4.jp/evaluation/maps/753?project_id=prd_jt

Let's focus on a point around the center

![image](tutorial_docs/odaiba_map_example.png)

## Setting the Environment after build and install

In [1]:
## source setup.bash after build and install
!source setup.bash

Environment setup complete:
PYTHONPATH = /home/ukenryu/map_test/lanelet2_extension/install/lib/python3/dist-packages:/home/ukenryu/map_test/lanelet2_extension/install/lib/python3/dist-packages:/home/ukenryu/map_test/lanelet2_extension/install/lib/python3/dist-packages:
LD_LIBRARY_PATH = /home/ukenryu/map_test/lanelet2_extension/install/lib:/usr/local/cuda/lib64:/home/ukenryu/map_test/lanelet2_extension/install/lib:/usr/local/cuda/lib64:
PATH = /home/ukenryu/map_test/lanelet2_extension/install/lib:/home/ukenryu/.easy_autoware/scenario_sim/scripts:/home/ukenryu/.easy_autoware/scripts:/usr/local/cuda/bin:/home/ukenryu/map_test/lanelet2_extension/install/lib:/home/ukenryu/.easy_autoware/scenario_sim/scripts:/home/ukenryu/.easy_autoware/scripts:/usr/local/cuda/bin:/home/ukenryu/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin:/snap/bin:/home/ukenryu/.local/bin:/home/ukenryu/.local/bin:/home/ukenryu/.local/bin:/home/ukenryu/.vscode

### Package Loading

In [2]:
from autoware_lanelet2_extension_python.projection import MGRSProjector
import autoware_lanelet2_extension_python.utility.query as query
import autoware_lanelet2_extension_python.utility.utilities as utilities
from autoware_lanelet2_extension_python.regulatory_elements import *
import lanelet2

<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<lanelet::ConstLineString3d, std::allocator<lanelet::ConstLineString3d> > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<lanelet::ConstPolygon3d, std::allocator<lanelet::ConstPolygon3d> > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<lanelet::ConstLanelet, std::allocator<lanelet::ConstLanelet> > already registered; second conversion method ignored.


## Standard Map loading

In [3]:
# MGRS Projector will be the default projector without a projector into file
projector = MGRSProjector(lanelet2.io.Origin(0.0, 0.0))

In [4]:
## PLEASE Change the path to your path
lanelet2_path = "/home/ukenryu/autoware_map/odaiba_stable/lanelet2_map.osm"

In [5]:
# Use the standard IO API for map loading
odaiba_map = lanelet2.io.load(lanelet2_path, projector)

## Getting Neighbourding Maps

In [6]:
query.getCurrentLanelets?

Signature: query.getCurrentLanelets(lanelets, point_or_pose)
Docstring:
Get current lanelets containing point/pose.

Args:
    lanelets: List of lanelets to search
    point_or_pose: numpy array [x,y,z] or [x,y,z,yaw]
File:      ~/map_test/lanelet2_extension/install/lib/python3/dist-packages/autoware_lanelet2_extension_python/utility/query.py
Type:      function


In [7]:
query.getLaneletsWithinRange?

Signature: query.getLaneletsWithinRange(lanelets, point, rng)
Docstring:
Get lanelets within range of a point.

Args:
    lanelets: List of lanelets to search
    point: Either lanelet2.core.BasicPoint2d or numpy array [x,y,z]
    rng: Search radius
File:      ~/map_test/lanelet2_extension/install/lib/python3/dist-packages/autoware_lanelet2_extension_python/utility/query.py
Type:      function


In [8]:
import numpy as np
search_point = np.array([89179, 43193, 0]) # we select the point [x, y, z] as indicated in the image above

In [9]:
# Get only the lanelet with sub_type==road
all_lanelets = query.laneletLayer(odaiba_map)
road_lanelets = query.roadLanelets(all_lanelets)


current_lanelets = query.getCurrentLanelets(road_lanelets, search_point)
len(current_lanelets), current_lanelets[0].id

(1, 126)

#### Benchmarking map extraction

`query.getLaneletsWithinRange` implemented by TIER IV is actually not efficient. It loops through the lanelets and to see if it is close enough to the search point.

`laneletLayer.search` utilizes R-tree data structure that significantly boost the search efficiency from O(n) to O(log(n)).

The results will be **a little bit different** because bounding box search contains a large area than the circular search.

In [10]:
%%timeit
# The point and search radius will automatically convert to the target type (double).
# So no need to handle the type too carefully here

search_range = 80
nearby_lanelets = query.getLaneletsWithinRange(road_lanelets, search_point, search_range)


56 ms ± 454 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [11]:
search_range = 80
nearby_lanelets = query.getLaneletsWithinRange(road_lanelets, search_point, search_range)
print(len(nearby_lanelets))

65


In [12]:
%%timeit
bounding_box = lanelet2.core.BoundingBox2d(
    lanelet2.core.BasicPoint2d(float(search_point[0] - search_range), float(search_point[1] - search_range)),
    lanelet2.core.BasicPoint2d(float(search_point[0] + search_range), float(search_point[1] + search_range))
)
nearby_lanelets = odaiba_map.laneletLayer.search(bounding_box)
nearby_lanelets = [lanelet for lanelet in nearby_lanelets if lanelet.attributes['subtype'] == 'road']
len(nearby_lanelets)

120 µs ± 795 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [13]:
bounding_box = lanelet2.core.BoundingBox2d(
    lanelet2.core.BasicPoint2d(float(search_point[0] - search_range), float(search_point[1] - search_range)),
    lanelet2.core.BasicPoint2d(float(search_point[0] + search_range), float(search_point[1] + search_range))
)
nearby_lanelets = odaiba_map.laneletLayer.search(bounding_box)
nearby_lanelets = [lanelet for lanelet in nearby_lanelets if lanelet.attributes['subtype'] == 'road']
print(len(nearby_lanelets))

74


### Visualize the search results

#### Helper functions (feel free to fold it after run)

In [14]:
## Define some common helper functions
import matplotlib.pyplot as plt
def plot_ll2_id(ll2, ax, text):
    xs, ys = np.array([pt.x for pt in ll2.centerline]), np.array([pt.y for pt in ll2.centerline])
    x, y = np.average(xs), np.average(ys)
    ax.text(x, y, text)


def plot_linestring(linestring, ax:plt.Axes, color, linestyle, **kwargs):
    xs = [pt.x for pt in linestring]
    ys = [pt.y for pt in linestring]
    ax.plot(xs, ys, color=color, linestyle=linestyle, **kwargs)

def compute_direction(lanelet):
    # compute with centerline if possible
    centerline = lanelet.centerline
    if len(centerline) < 2 or centerline.id==0:
        # compute with leftBound if possible
        leftBound = lanelet.leftBound
        if len(leftBound) < 2:
            # compute with rightBound
            rightBound = lanelet.rightBound
            if len(rightBound) < 2:
                # direction is not valid
                return 999
            p0 = rightBound[0]
            p1 = rightBound[-1]
            return np.arctan2(p1.y - p0.y, p1.x - p0.x)
        p0 = leftBound[0]
        p1 = leftBound[-1]
        #print(f"{lanelet.id} leftBound p0=({p0.x}, {p0.y}); p1=({p1.x}, {p1.y})")
        return np.arctan2(p1.y - p0.y, p1.x - p0.x)

    p0 = centerline[0]
    p1 = centerline[-1]
    return np.arctan2(p1.y - p0.y, p1.x - p0.x)

def visualize_lanelet_direction(lanelet, ax:plt.Axes):
    direction = compute_direction(lanelet)
    if abs(direction) > 2 * np.pi:
        # direction is not valid
        return
    
    arrow_length = 5
    # if centerline exist draw in centerline points
    ## We need to point out that many maps in TIER IV Autoware does not have centerline, so always remember that
    centerline = lanelet.centerline
    if len(centerline) > 1 and centerline.id>0:

        p0 = centerline[len(centerline) // 2]
        ax.arrow(p0.x, p0.y,
                 arrow_length * np.cos(direction), arrow_length * np.sin(direction), head_width=2, head_length=2, fc='k', ec='k')
        return
    # if centerline not exist draw in the average of leftBound rightBound points
    leftBound = lanelet.leftBound
    rightBound = lanelet.rightBound
    if len(leftBound) > 1 and len(rightBound) > 1:
        left_p0 = leftBound[len(leftBound) // 2]
        right_p0 = rightBound[len(rightBound) // 2]
        ax.arrow((left_p0.x + right_p0.x) / 2, (left_p0.y + right_p0.y) / 2,
                     arrow_length * np.cos(direction), arrow_length * np.sin(direction),
                      head_width=2, head_length=2, fc='k', ec='k')
        return

#### Draw the Neightbouring

In [15]:
%matplotlib qt

fig, ax = plt.subplots()

# draw search point
ax.scatter(search_point[0], search_point[1], color="black", marker="x", label="search point")

for lanelet in nearby_lanelets:
    plot_ll2_id(lanelet, ax, str(lanelet.id))
    # plot_linestring(lanelet.centerline, ax, "blue", "-", "centerline")
    plot_linestring(lanelet.leftBound, ax, "red", "-", label=f"leftBound_{lanelet.id}")
    plot_linestring(lanelet.rightBound, ax, "green", "-", label=f"rightBound_{lanelet.id}")
    visualize_lanelet_direction(lanelet, ax)

ax.set_aspect('equal')
# ax.legend()

The resulting image should look very similar to the one we have above

## Get Connections and Routing

In [16]:
# use lanelet2 standard APIs for getting
# We only run routing once after we load the entire map
traffic_rules = lanelet2.traffic_rules.create(
    lanelet2.traffic_rules.Locations.Germany,
    lanelet2.traffic_rules.Participants.Vehicle,
)
routing_graph = lanelet2.routing.RoutingGraph(odaiba_map, traffic_rules)

We know the current lanelet id is 126;  we can obtain the the lanelet with id=126, and also its following / preceding lanelets

In [17]:
# lanelet_126 = odaiba_map.laneletLayer.get(126)
lanelet_126 = current_lanelets[0]

In [18]:
# Get succeeding lanelets and preceding lanelets, which will also be very fast
succeeding_lanlets = routing_graph.following(lanelet_126)
preceding_lanlets = routing_graph.previous(lanelet_126)

In [19]:
fig, ax = plt.subplots()

# draw search point
ax.scatter(search_point[0], search_point[1], color="black", marker="x", label="search point")

for lanelet in succeeding_lanlets:
    plot_ll2_id(lanelet, ax, str(lanelet.id))
    # plot_linestring(lanelet.centerline, ax, "blue", "-", "centerline")
    plot_linestring(lanelet.leftBound, ax, "green", "-", label=f"leftBound_{lanelet.id}")
    plot_linestring(lanelet.rightBound, ax, "green", "-", label=f"rightBound_{lanelet.id}")
    visualize_lanelet_direction(lanelet, ax)

for lanelet in preceding_lanlets:
    plot_ll2_id(lanelet, ax, str(lanelet.id))
    # plot_linestring(lanelet.centerline, ax, "blue", "-", "centerline")
    plot_linestring(lanelet.leftBound, ax, "red", "-", label=f"leftBound_{lanelet.id}")
    plot_linestring(lanelet.rightBound, ax, "red", "-", label=f"rightBound_{lanelet.id}")
    visualize_lanelet_direction(lanelet, ax)

ax.set_aspect('equal')

### Example: Get Complete lanelet segments in the neighbouring

In [20]:
def find_all_movable_combinations(routing_graph, neighborhood_lanelets):
    """
    Find all possible combinations of movable lanelets.
    Args:
        routing_graph: Routing graph of the map
        neighborhood_lanelets: List of lanelets within a certain distance of the current lanelet
        search_distance: Optional parameter to limit path length (if None, only limit by max_path_length)
    Returns:
        List of complete lanelet paths (each path is a list of lanelets)
    
    First, construct a directed graph, where each node is a lanelet and each edge represents a possible transition between lanelets.
    Then, find all paths in this graph (including possible loops).
    """
    # Construct
    no_child_points = []
    children = {}
    no_parents_points = []
    parents = {}
    for lanelet in neighborhood_lanelets:
        children[lanelet] = []
        for succ in routing_graph.following(lanelet):
            if succ in neighborhood_lanelets:  # Only consider successors within the neighborhood
                children[lanelet].append(succ)
        if len(children[lanelet]) == 0:
            no_child_points.append(lanelet)
        parents[lanelet] = []
        for pred in routing_graph.previous(lanelet):
            if pred in neighborhood_lanelets:  # Only consider predecessors within the neighborhood
                parents[lanelet].append(pred)
        if len(parents[lanelet]) == 0:
            no_parents_points.append(lanelet)

    ## Find all possible paths from any of the no_child_points to any of the no_parents_points
    # Initialize a dictionary to store the paths from each no_child_point to each no_parents_point
    paths = []
    for start_point in no_parents_points:
        temp_stack = []
        def dfs():
            stack_last = temp_stack[-1]
            if stack_last in no_child_points:
                paths.append(temp_stack[:])
                return
            for child in children[stack_last]:
                temp_stack.append(child)
                dfs()
                temp_stack.pop()
        temp_stack.append(start_point)
        dfs()
    return paths

In [21]:
all_movable_combinations = find_all_movable_combinations(routing_graph, nearby_lanelets)

fig, ax = plt.subplots()
colors = plt.cm.tab10.colors
for i, combination in enumerate(all_movable_combinations):
    color = colors[i % len(colors)]
    lanelet = utilities.combineLaneletsShape(combination)
    line_strings = lanelet.polygon3d().lineStrings()
    for linestring in line_strings:
        plot_linestring(linestring, ax, color, "-")
    print(f"Movable combination: {[ll.id for ll in combination]}")
ax.set_aspect('equal')

Movable combination: [1471, 1394]
Movable combination: [1471, 1404]
Movable combination: [1472, 1395]
Movable combination: [1474, 1397, 1477, 1017, 126, 785, 132]
Movable combination: [1473, 1396, 1476, 1019, 1483]
Movable combination: [1473, 1396, 1476, 1018, 1484]
Movable combination: [1473, 1405, 1475, 1183, 1482]
Movable combination: [1363, 1489, 1492, 1495, 1176, 126, 785, 132]
Movable combination: [1402, 1476, 1019, 1483]
Movable combination: [1402, 1476, 1018, 1484]
Movable combination: [1401, 1475, 1183, 1482]
Movable combination: [1403, 1477, 1017, 126, 785, 132]
Movable combination: [1490, 1493, 1496, 1180, 1478, 1398]
Movable combination: [1491, 1494, 1498, 1015, 1480, 1400]
Movable combination: [1491, 1514, 1497, 1181, 1479, 1399]
Movable combination: [1491, 1514, 1497, 1181, 1479, 1407]
Movable combination: [1491, 1514, 1497, 1181, 1479, 1409]
Movable combination: [1491, 1514, 1497, 1181, 1479, 1408]
Movable combination: [324, 1178, 128, 130]
Movable combination: [324, 117